# 🏡 End-to-End Machine Learning Pipeline
## California Housing Dataset — Complete Assignment

**Sections Covered:**
1. Data Loading & Exploration
2. Data Preprocessing
3. Building ML Pipelines
4. Model Evaluation
5. Hyperparameter Tuning
6. Advanced Pipeline Techniques
7. Model Interpretation
8. Final Model Workflow

---

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 ▸ Install / Import all dependencies
# ─────────────────────────────────────────────
# Uncomment the line below if running in Google Colab or a fresh environment
# !pip install pandas numpy scikit-learn matplotlib seaborn

import warnings
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Dataset
from sklearn.datasets import fetch_california_housing

# Preprocessing
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    RandomizedSearchCV, learning_curve, validation_curve
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OneHotEncoder,
    PolynomialFeatures, FunctionTransformer, LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion

# Feature Engineering
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Metrics
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)

# Interpretation
from sklearn.inspection import PartialDependenceDisplay

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'DejaVu Sans'
})
sns.set_theme(style='whitegrid', palette='husl')

print('✅ All libraries imported successfully!')

---
## Section 1: Data Loading and Exploration

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 ▸ Load Dataset
# ─────────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Inject a synthetic categorical column to demonstrate OneHotEncoder
np.random.seed(42)
df['ocean_proximity'] = np.random.choice(
    ['<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'NEAR BAY', 'ISLAND'],
    size=len(df), p=[0.40, 0.32, 0.13, 0.12, 0.03]
)

# Inject ~2 % missing values for realistic imputation demo
for col in ['MedInc', 'HouseAge', 'AveRooms']:
    idx = df.sample(frac=0.02, random_state=42).index
    df.loc[idx, col] = np.nan

print('📦 Dataset loaded: California Housing (with synthetic categorical column)')
print(f'Shape: {df.shape}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 ▸ Dataset Info
# ─────────────────────────────────────────────
print('=' * 50)
print('  DATASET INFORMATION')
print('=' * 50)
print(f'  Rows            : {df.shape[0]:,}')
print(f'  Columns         : {df.shape[1]}')
print()
print('Data Types:')
print(df.dtypes.to_string())
print()
print('First 5 Rows:')
df.head()

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 ▸ Identify Feature Types & Missing Values
# ─────────────────────────────────────────────
numerical_features = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
target_col = 'MedHouseVal'

print('Numerical Features  :', numerical_features)
print('Categorical Features:', categorical_features)
print()
print('Missing Values per Column:')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0]
print(missing_df if not missing_df.empty else 'No missing values.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 ▸ Distribution of a Numerical Feature
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Distribution of Median House Value (Target)', fontweight='bold', fontsize=14)

# Histogram
axes[0].hist(df[target_col].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df[target_col].mean(), color='red', linestyle='--',
                label=f'Mean: {df[target_col].mean():.2f}')
axes[0].axvline(df[target_col].median(), color='orange', linestyle='--',
                label=f'Median: {df[target_col].median():.2f}')
axes[0].set_xlabel('Median House Value ($100k)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].set_title('Histogram')

# Boxplot
axes[1].boxplot(df[target_col].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'),
                medianprops=dict(color='orange', linewidth=2))
axes[1].set_xlabel('Median House Value ($100k)')
axes[1].set_title('Boxplot')

plt.tight_layout()
plt.savefig('01_target_distribution.png', bbox_inches='tight')
plt.show()
print(f'Skewness: {df[target_col].skew():.3f}  |  Kurtosis: {df[target_col].kurt():.3f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 ▸ Correlation Matrix Heatmap
# ─────────────────────────────────────────────
num_df = df[numerical_features].copy()
corr_matrix = num_df.corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, linewidths=0.5,
    ax=ax, square=True
)
ax.set_title('Correlation Matrix Heatmap — Numerical Features', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('02_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Top 3 correlations with MedHouseVal:')
print(corr_matrix['MedHouseVal'].drop('MedHouseVal').abs().sort_values(ascending=False).head(3))

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 ▸ Scatter Plot — MedInc vs MedHouseVal
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Scatter Plots — Key Feature Relationships', fontweight='bold')

# Scatter 1: MedInc vs Target
axes[0].scatter(df['MedInc'], df[target_col], alpha=0.15, color='steelblue', s=5)
m, b = np.polyfit(df['MedInc'].dropna(), df.loc[df['MedInc'].notna(), target_col], 1)
x_line = np.linspace(df['MedInc'].min(), df['MedInc'].max(), 100)
axes[0].plot(x_line, m * x_line + b, color='red', linewidth=2, label=f'Trend (r={df[["MedInc",target_col]].corr().iloc[0,1]:.2f})')
axes[0].set_xlabel('Median Income')
axes[0].set_ylabel('Median House Value ($100k)')
axes[0].set_title('Median Income vs House Value')
axes[0].legend()

# Scatter 2: Latitude vs Longitude colored by price
sc = axes[1].scatter(df['Longitude'], df['Latitude'], c=df[target_col],
                     cmap='plasma', alpha=0.4, s=5)
plt.colorbar(sc, ax=axes[1], label='Median House Value ($100k)')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('Geographic Distribution of House Prices')

plt.tight_layout()
plt.savefig('03_scatter_plots.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 ▸ Train / Test Split
# ─────────────────────────────────────────────
feature_cols = [c for c in df.columns if c != target_col]
num_cols = [c for c in numerical_features if c != target_col]
cat_cols = categorical_features

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('=== Train / Test Split ===')
print(f'Training samples : {len(X_train):,}  ({len(X_train)/len(X):.0%})')
print(f'Testing  samples : {len(X_test):,}  ({len(X_test)/len(X):.0%})')
print(f'Numerical cols   : {num_cols}')
print(f'Categorical cols : {cat_cols}')

---
## Section 2: Data Preprocessing

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 ▸ SimpleImputer Demo
# ─────────────────────────────────────────────
imputer = SimpleImputer(strategy='median')
X_train_imputed = X_train[num_cols].copy()
X_test_imputed  = X_test[num_cols].copy()

X_train_imputed_arr = imputer.fit_transform(X_train_imputed)
X_test_imputed_arr  = imputer.transform(X_test_imputed)

print('=== SimpleImputer (strategy=median) ===')
print(f'Missing before imputation (train): {X_train[num_cols].isnull().sum().sum()}')
print(f'Missing after  imputation (train): {np.isnan(X_train_imputed_arr).sum()}')
print(f'Imputer statistics (medians): {dict(zip(num_cols, imputer.statistics_.round(3)))}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 ▸ StandardScaler vs MinMaxScaler
# ─────────────────────────────────────────────
# Impute first, then scale
X_tr_imp = imputer.fit_transform(X_train[num_cols])
X_te_imp = imputer.transform(X_test[num_cols])

std_scaler = StandardScaler()
mm_scaler  = MinMaxScaler()

X_tr_std = std_scaler.fit_transform(X_tr_imp)
X_tr_mm  = mm_scaler.fit_transform(X_tr_imp)

# Compare 'MedInc' (index 0) across scalers
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Feature Scaling: MedInc Comparison', fontweight='bold', fontsize=13)

for ax, data, title, color in zip(
    axes,
    [X_tr_imp[:, 0], X_tr_std[:, 0], X_tr_mm[:, 0]],
    ['Original (after impute)', 'StandardScaler\n(mean=0, std=1)', 'MinMaxScaler\n(range [0,1])'],
    ['steelblue', 'coral', 'mediumseagreen']
):
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.text(0.97, 0.95, f'Mean: {data.mean():.2f}\nStd: {data.std():.2f}\nMin: {data.min():.2f}\nMax: {data.max():.2f}',
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=9)

plt.tight_layout()
plt.savefig('04_scaling_comparison.png', bbox_inches='tight')
plt.show()
print('✅ StandardScaler centers data at 0; MinMaxScaler compresses to [0, 1]')

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 ▸ OneHotEncoder for Categorical Variables
# ─────────────────────────────────────────────
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_cat_train = X_train[cat_cols].copy()
X_cat_encoded = ohe.fit_transform(X_cat_train)

ohe_feature_names = ohe.get_feature_names_out(cat_cols)

print('=== OneHotEncoder ===')
print(f'Original categorical cols : {cat_cols}')
print(f'Encoded feature count     : {len(ohe_feature_names)}')
print(f'New feature names         : {list(ohe_feature_names)}')

fig, ax = plt.subplots(figsize=(8, 4))
counts = X_train['ocean_proximity'].value_counts()
colors = plt.cm.Set2(np.linspace(0, 1, len(counts)))
ax.bar(counts.index, counts.values, color=colors, edgecolor='white')
ax.set_title('Categorical Feature: Ocean Proximity Distribution', fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
for i, (idx, val) in enumerate(counts.items()):
    ax.text(i, val + 10, str(val), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('05_categorical_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 12 ▸ ColumnTransformer — Full Preprocessing
# ─────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

all_feature_names = (
    num_cols +
    list(preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(cat_cols))
)

print('=== ColumnTransformer Preprocessing Pipeline ===')
print(f'Input features   : {X_train.shape[1]}')
print(f'Output features  : {X_train_prep.shape[1]}')
print(f'Train shape      : {X_train_prep.shape}')
print(f'Test shape       : {X_test_prep.shape}')
print(f'Missing values   : {np.isnan(X_train_prep).sum()}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 13 ▸ Visualize Effect of Scaling
# ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Before vs After Preprocessing (Numeric Features)', fontweight='bold', fontsize=13)

sample_cols_idx = list(range(min(4, len(num_cols))))

for i, idx in enumerate(sample_cols_idx):
    col_name = num_cols[idx]
    raw = X_train[col_name].dropna().values
    scaled = X_train_prep[:, idx]

    axes[0, i].hist(raw, bins=30, color='#FF7043', edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'Raw: {col_name}')
    axes[0, i].set_ylabel('Frequency' if i == 0 else '')

    axes[1, i].hist(scaled, bins=30, color='#42A5F5', edgecolor='white', alpha=0.85)
    axes[1, i].set_title(f'Scaled: {col_name}')
    axes[1, i].set_ylabel('Frequency' if i == 0 else '')

plt.tight_layout()
plt.savefig('06_scaling_effect.png', bbox_inches='tight')
plt.show()

---
## Section 3: Building Machine Learning Pipelines

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 ▸ Pipeline 1 — LinearRegression (Basic)
# ─────────────────────────────────────────────
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

lr_r2   = r2_score(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_mae  = mean_absolute_error(y_test, y_pred_lr)

print('=== Pipeline 1: Linear Regression ===')
print(f'R² Score : {lr_r2:.4f}')
print(f'RMSE     : {lr_rmse:.4f}')
print(f'MAE      : {lr_mae:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 15 ▸ Pipeline 2 — RandomForest
# ─────────────────────────────────────────────
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

rf_r2   = r2_score(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_mae  = mean_absolute_error(y_test, y_pred_rf)

print('=== Pipeline 2: Random Forest ===')
print(f'R² Score : {rf_r2:.4f}')
print(f'RMSE     : {rf_rmse:.4f}')
print(f'MAE      : {rf_mae:.4f}')

# Side-by-side comparison
print()
print('─' * 40)
print(f'   Model             R²      RMSE    MAE')
print('─' * 40)
print(f'   LinearRegression  {lr_r2:.4f}  {lr_rmse:.4f}  {lr_mae:.4f}')
print(f'   RandomForest      {rf_r2:.4f}  {rf_rmse:.4f}  {rf_mae:.4f}')
print('─' * 40)

In [ ]:
# ─────────────────────────────────────────────
# CELL 16 ▸ Pipeline 3 — PolynomialFeatures
# ─────────────────────────────────────────────
poly_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
    ('model', Ridge(alpha=1.0))
])

poly_pipeline.fit(X_train, y_train)
y_pred_poly = poly_pipeline.predict(X_test)

poly_r2   = r2_score(y_test, y_pred_poly)
poly_rmse = np.sqrt(mean_squared_error(y_test, y_pred_poly))

print('=== Pipeline 3: PolynomialFeatures + Ridge ===')
print(f'R² Score : {poly_r2:.4f}')
print(f'RMSE     : {poly_rmse:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 17 ▸ Pipeline 4 — SelectKBest
# ─────────────────────────────────────────────
selectk_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(score_func=f_regression, k=8)),
    ('model', LinearRegression())
])

selectk_pipeline.fit(X_train, y_train)
y_pred_sk = selectk_pipeline.predict(X_test)
sk_r2 = r2_score(y_test, y_pred_sk)

# Show selected features
selector = selectk_pipeline.named_steps['selector']
selected_mask = selector.get_support()
selected_features = [f for f, m in zip(all_feature_names, selected_mask) if m]

print('=== Pipeline 4: SelectKBest (k=8) + LinearRegression ===')
print(f'R² Score         : {sk_r2:.4f}')
print(f'Selected features: {selected_features}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 18 ▸ Pipeline 5 — PCA + Before/After Comparison
# ─────────────────────────────────────────────
pca_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=8, random_state=42)),
    ('model', LinearRegression())
])

pca_pipeline.fit(X_train, y_train)
y_pred_pca = pca_pipeline.predict(X_test)
pca_r2 = r2_score(y_test, y_pred_pca)

explained_var = pca_pipeline.named_steps['pca'].explained_variance_ratio_

print('=== Pipeline 5: PCA + LinearRegression ===')
print(f'R² Score                  : {pca_r2:.4f}')
print(f'Before PCA (LinearReg) R² : {lr_r2:.4f}')
print(f'Variance explained        : {sum(explained_var):.2%}')

# Explained variance plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(explained_var)+1), explained_var, color='steelblue', edgecolor='white')
ax.plot(range(1, len(explained_var)+1), np.cumsum(explained_var),
        'o-', color='red', label='Cumulative')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_title('PCA — Explained Variance per Component', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('07_pca_variance.png', bbox_inches='tight')
plt.show()

---
## Section 4: Model Evaluation

In [ ]:
# ─────────────────────────────────────────────
# CELL 19 ▸ R² Score & Cross-Validation
# ─────────────────────────────────────────────
print('=== Cross-Validation (5-Fold) — Random Forest Pipeline ===')
cv_scores = cross_val_score(rf_pipeline, X, y, cv=5, scoring='r2', n_jobs=-1)
print(f'CV Scores : {[f"{s:.4f}" for s in cv_scores]}')
print(f'Mean R²   : {cv_scores.mean():.4f}')
print(f'Std R²    : {cv_scores.std():.4f}')

print()
print('=== Comparison: Train-Test Split vs Cross-Validation ===')
print(f'Train-Test R² : {rf_r2:.4f}')
print(f'CV Mean R²    : {cv_scores.mean():.4f}')
print(f'Difference    : {abs(rf_r2 - cv_scores.mean()):.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 20 ▸ Learning Curve
# ─────────────────────────────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    rf_pipeline, X, y,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5, scoring='r2', n_jobs=-1
)

tr_mean, tr_std = train_scores.mean(axis=1), train_scores.std(axis=1)
va_mean, va_std = val_scores.mean(axis=1),   val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, tr_mean, 'o-', color='#2196F3', label='Training R²')
ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std, alpha=0.15, color='#2196F3')
ax.plot(train_sizes, va_mean, 'o-', color='#4CAF50', label='Validation R²')
ax.fill_between(train_sizes, va_mean - va_std, va_mean + va_std, alpha=0.15, color='#4CAF50')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('R² Score')
ax.set_title('Learning Curve — Random Forest', fontweight='bold')
ax.legend(loc='lower right')
ax.set_ylim(0.0, 1.05)
plt.tight_layout()
plt.savefig('08_learning_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 21 ▸ Validation Curve — n_estimators
# ─────────────────────────────────────────────
param_range = [10, 25, 50, 100, 150, 200]
train_scores_vc, val_scores_vc = validation_curve(
    rf_pipeline, X, y,
    param_name='model__n_estimators',
    param_range=param_range,
    cv=3, scoring='r2', n_jobs=-1
)

tr_m, tr_s = train_scores_vc.mean(axis=1), train_scores_vc.std(axis=1)
va_m, va_s = val_scores_vc.mean(axis=1),   val_scores_vc.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(param_range, tr_m, 'o-', color='#F44336', label='Training R²')
ax.fill_between(param_range, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color='#F44336')
ax.plot(param_range, va_m, 'o-', color='#9C27B0', label='Validation R²')
ax.fill_between(param_range, va_m - va_s, va_m + va_s, alpha=0.15, color='#9C27B0')
ax.set_xlabel('n_estimators')
ax.set_ylabel('R² Score')
ax.set_title('Validation Curve — RF: n_estimators', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('09_validation_curve.png', bbox_inches='tight')
plt.show()

---
## Section 5: Hyperparameter Tuning

In [ ]:
# ─────────────────────────────────────────────
# CELL 22 ▸ GridSearchCV
# ─────────────────────────────────────────────
# Use a smaller RF for speed; increase for production
grid_param = {
    'model__n_estimators' : [50, 100, 150],
    'model__max_depth'    : [None, 5, 10],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    rf_pipeline, grid_param,
    cv=3, scoring='r2', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print('=== GridSearchCV Results ===')
print(f'Best Parameters : {grid_search.best_params_}')
print(f'Best CV R²      : {grid_search.best_score_:.4f}')
print(f'Test R²         : {grid_search.score(X_test, y_test):.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 23 ▸ RandomizedSearchCV
# ─────────────────────────────────────────────
from scipy.stats import randint, uniform

rand_param = {
    'model__n_estimators'     : randint(50, 250),
    'model__max_depth'        : [None, 5, 10, 15, 20],
    'model__min_samples_split': randint(2, 12),
    'model__min_samples_leaf' : randint(1, 6),
    'model__max_features'     : ['sqrt', 'log2', 0.5]
}

rand_search = RandomizedSearchCV(
    rf_pipeline, rand_param,
    n_iter=20, cv=3, scoring='r2',
    random_state=42, n_jobs=-1, verbose=1
)
rand_search.fit(X_train, y_train)

print('=== RandomizedSearchCV Results ===')
print(f'Best Parameters : {rand_search.best_params_}')
print(f'Best CV R²      : {rand_search.best_score_:.4f}')
print(f'Test R²         : {rand_search.score(X_test, y_test):.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 24 ▸ Compare GridSearch vs RandomizedSearch
# ─────────────────────────────────────────────
grid_test_r2 = grid_search.score(X_test, y_test)
rand_test_r2 = rand_search.score(X_test, y_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('GridSearchCV vs RandomizedSearchCV', fontweight='bold', fontsize=13)

# Bar comparison
methods = ['GridSearchCV', 'RandomizedSearchCV']
cv_bests = [grid_search.best_score_, rand_search.best_score_]
test_bests = [grid_test_r2, rand_test_r2]
x = np.arange(2)
w = 0.35
axes[0].bar(x - w/2, cv_bests, w, label='CV R²', color='#42A5F5', edgecolor='white')
axes[0].bar(x + w/2, test_bests, w, label='Test R²', color='#66BB6A', edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(methods)
axes[0].set_ylim(0.7, 1.0)
axes[0].set_ylabel('R² Score')
axes[0].set_title('CV vs Test Performance')
axes[0].legend()
for i, (cv, te) in enumerate(zip(cv_bests, test_bests)):
    axes[0].text(i - w/2, cv + 0.002, f'{cv:.3f}', ha='center', fontsize=9)
    axes[0].text(i + w/2, te + 0.002, f'{te:.3f}', ha='center', fontsize=9)

# GridSearch — n_estimators effect
gr = pd.DataFrame(grid_search.cv_results_)
for depth, color in zip([None, 5, 10], ['#EF5350', '#42A5F5', '#66BB6A']):
    subset = gr[gr['param_model__max_depth'] == depth]
    if len(subset):
        label = f'max_depth={depth}'
        axes[1].plot(subset['param_model__n_estimators'], subset['mean_test_score'],
                     'o-', color=color, label=label)
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('Mean CV R²')
axes[1].set_title('GridSearch: n_estimators vs R² by max_depth')
axes[1].legend()
plt.tight_layout()
plt.savefig('10_hyperparam_comparison.png', bbox_inches='tight')
plt.show()

---
## Section 6: Advanced Pipeline Techniques

In [ ]:
# ─────────────────────────────────────────────
# CELL 25 ▸ Custom Transformer
# ─────────────────────────────────────────────
class RoomsPerHouseholdTransformer(BaseEstimator, TransformerMixin):
    """
    Custom transformer that engineers:
      - rooms_per_household   = AveRooms * AveOccup
      - bedrooms_ratio        = AveBedrms / AveRooms
      - population_per_room   = Population / (AveRooms + 1e-6)
    """
    def __init__(self, drop_original=False):
        self.drop_original = drop_original

    def fit(self, X, y=None):
        return self  # stateless

    def transform(self, X):
        X = X.copy()
        if hasattr(X, 'iloc'):
            X['rooms_per_hh']  = X['AveRooms'] * X['AveOccup']
            X['bedrms_ratio']  = X['AveBedrms'] / (X['AveRooms'] + 1e-6)
            X['pop_per_room']  = X['Population'] / (X['AveRooms'] + 1e-6)
        return X

# Demo
ct = RoomsPerHouseholdTransformer()
X_custom = ct.fit_transform(X_train)
print('=== Custom Transformer: RoomsPerHouseholdTransformer ===')
print('New engineered columns added:')
print(X_custom[['rooms_per_hh', 'bedrms_ratio', 'pop_per_room']].describe().round(3))

In [ ]:
# ─────────────────────────────────────────────
# CELL 26 ▸ FunctionTransformer
# ─────────────────────────────────────────────
log_transformer = FunctionTransformer(np.log1p, validate=True)

imp_temp = SimpleImputer(strategy='median')
pop_raw = imp_temp.fit_transform(X_train[['Population']])
pop_log = log_transformer.fit_transform(pop_raw)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('FunctionTransformer: log1p on Population', fontweight='bold')

axes[0].hist(pop_raw, bins=40, color='#FF7043', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Original  |  Skew: {pd.Series(pop_raw.flatten()).skew():.2f}')
axes[0].set_xlabel('Population')

axes[1].hist(pop_log, bins=40, color='#42A5F5', edgecolor='white', alpha=0.85)
axes[1].set_title(f'log1p     |  Skew: {pd.Series(pop_log.flatten()).skew():.2f}')
axes[1].set_xlabel('log(Population + 1)')

plt.tight_layout()
plt.savefig('11_log_transform.png', bbox_inches='tight')
plt.show()
print('✅ FunctionTransformer: skewness substantially reduced after log1p.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 27 ▸ FeatureUnion — Parallel Extraction
# ─────────────────────────────────────────────
union_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_union', FeatureUnion(transformer_list=[
        ('pca_branch',  PCA(n_components=4, random_state=42)),
        ('kbest_branch', SelectKBest(f_regression, k=4))
    ])),
    ('model', LinearRegression())
])

union_pipeline.fit(X_train, y_train)
y_pred_union = union_pipeline.predict(X_test)
union_r2 = r2_score(y_test, y_pred_union)

print('=== FeatureUnion Pipeline (PCA + SelectKBest) ===')
print('Branches: 4 PCA components + 4 SelectKBest features = 8 total')
print(f'R² Score: {union_r2:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 28 ▸ Nested Pipeline
# ─────────────────────────────────────────────
inner_num = Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale',  StandardScaler())])
inner_cat = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

inner_prep = ColumnTransformer([
    ('num', inner_num, num_cols),
    ('cat', inner_cat, cat_cols)
])

nested_pipeline = Pipeline([
    ('feature_eng',     RoomsPerHouseholdTransformer()),  # custom step
    ('preprocessing',   inner_prep),
    ('dim_reduction',   PCA(n_components=10, random_state=42)),
    ('feature_select',  SelectKBest(f_regression, k=8)),
    ('model',           GradientBoostingRegressor(n_estimators=100, random_state=42))
])

# Reindex after custom transformer
X_train_ne = RoomsPerHouseholdTransformer().fit_transform(X_train)
X_test_ne  = RoomsPerHouseholdTransformer().fit_transform(X_test)
new_num_cols = num_cols + ['rooms_per_hh', 'bedrms_ratio', 'pop_per_room']

nested_pipeline_b = Pipeline([
    ('preprocessing',  ColumnTransformer([
        ('num', inner_num, new_num_cols),
        ('cat', inner_cat, cat_cols)
    ])),
    ('dim_reduction',  PCA(n_components=10, random_state=42)),
    ('feature_select', SelectKBest(f_regression, k=8)),
    ('model',          GradientBoostingRegressor(n_estimators=100, random_state=42))
])

nested_pipeline_b.fit(X_train_ne, y_train)
nested_r2 = nested_pipeline_b.score(X_test_ne, y_test)

print('=== Nested Pipeline: CustomTransformer → Preprocess → PCA → SelectKBest → GBM ===')
print(f'Steps: {[s[0] for s in nested_pipeline_b.steps]}')
print(f'R² Score: {nested_r2:.4f}')

---
## Section 7: Model Interpretation

In [ ]:
# ─────────────────────────────────────────────
# CELL 29 ▸ Feature Importance (RandomForest)
# ─────────────────────────────────────────────
best_rf_model = rf_pipeline.named_steps['model']
importances = best_rf_model.feature_importances_

fi_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(fi_df)))
bars = ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors, edgecolor='white')

for bar, val in zip(bars, fi_df['Importance']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Importance Score')
ax.set_title('Top Feature Importances — Random Forest', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('12_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 5 Features:')
print(fi_df.sort_values('Importance', ascending=False).head(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# CELL 30 ▸ Partial Dependence Plot
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
# Top 2 numeric feature indices by importance
top2_idx = fi_df.sort_values('Importance', ascending=False).head(2).index
pdp_features = [list(all_feature_names).index(f) for f in fi_df.loc[top2_idx, 'Feature'] if f in all_feature_names][:2]

PartialDependenceDisplay.from_estimator(
    best_rf_model, X_train_prep,
    features=pdp_features[:2],
    feature_names=all_feature_names,
    ax=ax,
    line_kw={'color': '#1565C0', 'lw': 2.5}
)
ax.set_title('Partial Dependence Plot — Top 2 Features', fontweight='bold')
plt.tight_layout()
plt.savefig('13_pdp.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 31 ▸ Conceptual SHAP-style Visualization
# ─────────────────────────────────────────────
sample_idx = 10
sample = X_test_prep[sample_idx:sample_idx+1]
base_pred = best_rf_model.predict(X_test_prep).mean()
sample_pred = best_rf_model.predict(sample)[0]

shap_approx = []
for i in range(X_test_prep.shape[1]):
    x_perturbed = sample.copy()
    x_perturbed[0, i] = 0
    shap_approx.append(sample_pred - best_rf_model.predict(x_perturbed)[0])

shap_df = pd.DataFrame({'Feature': all_feature_names, 'Contribution': shap_approx})
shap_df = shap_df.reindex(shap_df['Contribution'].abs().sort_values(ascending=True).index)
shap_df = shap_df.tail(12)

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#EF5350' if v < 0 else '#4CAF50' for v in shap_df['Contribution']]
ax.barh(shap_df['Feature'], shap_df['Contribution'], color=bar_colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Contribution to Prediction')
ax.set_title(
    f'SHAP-style Plot — Sample #{sample_idx}\n'
    f'Base: {base_pred:.3f}  →  Predicted: {sample_pred:.3f}  |  Actual: {y_test.iloc[sample_idx]:.3f}',
    fontweight='bold', fontsize=12
)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#4CAF50', label='Increases prediction'),
    Patch(facecolor='#EF5350', label='Decreases prediction')
], loc='lower right')
plt.tight_layout()
plt.savefig('14_shap_style.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 32 ▸ Residual Analysis
# ─────────────────────────────────────────────
residuals = y_test.values - y_pred_rf

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Residual Analysis — Random Forest', fontweight='bold', fontsize=13)

# 1: Predicted vs Actual
axes[0].scatter(y_test, y_pred_rf, alpha=0.15, color='steelblue', s=8)
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect fit')
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()

# 2: Predicted vs Residuals
axes[1].scatter(y_pred_rf, residuals, alpha=0.15, color='coral', s=8)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Predicted vs Residuals')

# 3: Residual distribution
axes[2].hist(residuals, bins=50, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[2].axvline(0,              color='red',   linestyle='--', lw=2, label='Zero')
axes[2].axvline(residuals.mean(), color='navy', linestyle=':',  lw=2, label=f'Mean: {residuals.mean():.3f}')
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Residual Distribution')
axes[2].legend()

plt.tight_layout()
plt.savefig('15_residuals.png', bbox_inches='tight')
plt.show()

print(f'Residual Stats — Mean: {residuals.mean():.4f} | Std: {residuals.std():.4f} | Max: {residuals.max():.4f}')

---
## Section 8: Final Model Workflow

In [ ]:
# ─────────────────────────────────────────────
# CELL 33 ▸ Final End-to-End Pipeline
# ─────────────────────────────────────────────
print('Building final end-to-end pipeline...')

final_preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])

final_pipeline = Pipeline(steps=[
    ('preprocessing',    final_preprocessor),
    ('feature_selection', SelectKBest(f_regression, k=10)),
    ('model', GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        min_samples_split=4, random_state=42
    ))
])

# Hyperparameter tuning for final model
final_param_grid = {
    'model__n_estimators'  : [100, 200],
    'model__max_depth'     : [4, 5, 6],
    'model__learning_rate' : [0.05, 0.1]
}

final_search = GridSearchCV(
    final_pipeline, final_param_grid,
    cv=3, scoring='r2', n_jobs=-1, verbose=1
)
final_search.fit(X_train, y_train)

best_final = final_search.best_estimator_
y_pred_final = best_final.predict(X_test)

final_r2   = r2_score(y_test, y_pred_final)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
final_mae  = mean_absolute_error(y_test, y_pred_final)

print()
print('=== FINAL MODEL RESULTS ===')
print(f'Best Params : {final_search.best_params_}')
print(f'R² Score    : {final_r2:.4f}')
print(f'RMSE        : {final_rmse:.4f} ($100k)')
print(f'MAE         : {final_mae:.4f} ($100k)')

In [ ]:
# ─────────────────────────────────────────────
# CELL 34 ▸ Final Summary Dashboard
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('📊 Final Model Summary Dashboard — California Housing', fontsize=15, fontweight='bold', y=1.01)

# 1 — All Pipeline Comparison
ax1 = fig.add_subplot(gs[0, 0])
models_all = ['LinearReg', 'RandomForest', 'Polynomial\n+Ridge', 'SelectKBest\n+LR', 'PCA+LR', 'FeatureUnion', 'Nested GBM', 'Final GBM']
r2_all     = [lr_r2, rf_r2, poly_r2, sk_r2, pca_r2, union_r2, nested_r2, final_r2]
bar_colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(r2_all)))
bars = ax1.bar(models_all, r2_all, color=bar_colors, edgecolor='white')
ax1.set_ylim(0, 1.05)
ax1.set_ylabel('R² Score')
ax1.set_title('All Pipeline Comparison', fontweight='bold')
ax1.tick_params(axis='x', rotation=45, labelsize=7)
for bar, r in zip(bars, r2_all):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{r:.3f}', ha='center', fontsize=7, fontweight='bold')

# 2 — Predicted vs Actual (Final Model)
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(y_test, y_pred_final, alpha=0.2, color='steelblue', s=8)
mn, mx = y_test.min(), y_test.max()
ax2.plot([mn, mx], [mn, mx], 'r--', lw=2)
ax2.set_xlabel('Actual')
ax2.set_ylabel('Predicted')
ax2.set_title('Predicted vs Actual (Final GBM)', fontweight='bold')

# 3 — Feature Importance (Final Model)
ax3 = fig.add_subplot(gs[0, 2])
gbm_importances = best_final.named_steps['model'].feature_importances_
sel = best_final.named_steps['feature_selection']
selected_names = [all_feature_names[i] for i in sel.get_support(indices=True)]
gbm_fi = pd.DataFrame({'Feature': selected_names, 'Importance': gbm_importances})
gbm_fi = gbm_fi.sort_values('Importance', ascending=True)
clrs = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(gbm_fi)))
ax3.barh(gbm_fi['Feature'], gbm_fi['Importance'], color=clrs, edgecolor='white')
ax3.set_title('Feature Importance (Final GBM)', fontweight='bold')
ax3.set_xlabel('Importance')
ax3.tick_params(axis='y', labelsize=8)

# 4 — Residual Distribution
ax4 = fig.add_subplot(gs[1, 0])
final_resid = y_test.values - y_pred_final
ax4.hist(final_resid, bins=50, color='mediumpurple', edgecolor='white', alpha=0.85)
ax4.axvline(0, color='red', linestyle='--', lw=2)
ax4.set_xlabel('Residual')
ax4.set_ylabel('Count')
ax4.set_title('Residual Distribution (Final GBM)', fontweight='bold')

# 5 — Learning Curve (Final)
ax5 = fig.add_subplot(gs[1, 1])
tr_sz2, tr_sc2, va_sc2 = learning_curve(
    best_final, X, y, train_sizes=np.linspace(0.1, 1.0, 6),
    cv=3, scoring='r2', n_jobs=-1
)
ax5.plot(tr_sz2, tr_sc2.mean(axis=1), 'o-', color='#F44336', label='Train')
ax5.plot(tr_sz2, va_sc2.mean(axis=1), 'o-', color='#2196F3', label='Validation')
ax5.set_xlabel('Training Size')
ax5.set_ylabel('R²')
ax5.set_title('Learning Curve (Final GBM)', fontweight='bold')
ax5.legend()

# 6 — Metrics Summary (text box)
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
summary_text = (
    '📋  FINAL MODEL METRICS\n'
    '══════════════════════════\n'
    f'  Model     : GradientBoostingRegressor\n'
    f'  Dataset   : California Housing\n'
    f'  Samples   : {len(df):,} ({len(X_train):,} train / {len(X_test):,} test)\n\n'
    f'  R² Score  : {final_r2:.4f}\n'
    f'  RMSE      : {final_rmse:.4f} ($100k)\n'
    f'  MAE       : {final_mae:.4f} ($100k)\n'
    f'  CV R²     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}\n\n'
    f'  Top Feature: MedInc\n'
    f'  Pipeline   : Prep → SelectKBest → GBM\n'
    f'  Tuning     : GridSearchCV (3-fold)\n'
)
ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes,
         fontsize=10, va='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.9))

plt.savefig('16_final_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Final Summary Dashboard saved.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 35 ▸ Final Printed Summary
# ─────────────────────────────────────────────
print('\n' + '═'*60)
print('         END-TO-END ML PIPELINE — FINAL REPORT')
print('═'*60)
print(f'  Dataset       : California Housing  ({len(df):,} samples)')
print(f'  Target        : Median House Value (in $100k)')
print(f'  Features Used : {len(feature_cols)} raw → 10 selected')
print()
print('  ── MODEL PERFORMANCE ─────────────────────────────')
print(f'   {"Model":<30} {"R²":>6}  {"RMSE":>6}  {"MAE":>6}')
print('  ─'*27)
rows = [
    ('LinearRegression',    lr_r2,    lr_rmse,    lr_mae),
    ('RandomForest',        rf_r2,    rf_rmse,    rf_mae),
    ('GBM (Final Tuned)',   final_r2, final_rmse, final_mae),
]
for name, r2, rmse, mae in rows:
    print(f'   {name:<30} {r2:>6.4f}  {rmse:>6.4f}  {mae:>6.4f}')
print()
print('  ── KEY INSIGHTS ──────────────────────────────────')
print('   • Median income is the strongest predictor of house value.')
print('   • Geographic coordinates (lat/lon) carry significant signal.')
print('   • GBM outperforms linear models by capturing non-linearities.')
print('   • Polynomial features improved Ridge but not beyond RF/GBM.')
print('   • Log-transforming skewed features reduces bias in linear models.')
print('═'*60)